# Alumni RAG Agent - CMU Africa Alumni Tracking and Support System

Our Agent complete workflow of the Alumni RAG Agent:
1. Environment setup and LangSmith tracing
2. Sample alumni data ingestion
3. **Automated Discovery** of alumni profiles via Tavily Search
4. Multi-step agent query execution
5. Verification and groundedness scoring

## 1. Environment Setup

In [3]:
import os
import sys

# Add parent directory to path for imports
sys.path.insert(0, os.path.dirname(os.getcwd()))

from dotenv import load_dotenv
load_dotenv()

# Enable LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "alumni-rag-agent-hw2"

# Verify environment
print("Environment Check:")
print(f"  MongoDB URI: {'Set Successful!' if os.environ.get('MONGODB_URI') else ' Missing'}")
print(f"  OpenAI API Key: {'Set Successful!' if os.environ.get('OPENAI_API_KEY') else 'Missing'}")
print(f"  LangSmith API Key: {'Set Successful!' if os.environ.get('LANGCHAIN_API_KEY') else ' Missing'}")
print(f" Tavily API Key: {"Set Successful!" if os.environ.get('TAVILY_API_KEY') else ' Missing '}")

Environment Check:
  MongoDB URI: Set Successful!
  OpenAI API Key: Set Successful!
  LangSmith API Key: Set Successful!
 Tavily API Key: Set Successful!


## 2. Initialize the Agent

In [2]:
from src.agent import AlumniAgent, SAMPLE_ALUMNI

# Initialize the agent (connects to MongoDB, sets up tools)
agent = AlumniAgent()

print(f"\nSample alumni data available: {len(SAMPLE_ALUMNI)} profiles")
for alumni in SAMPLE_ALUMNI:
    print(f"  - {alumni['name']} ({alumni['program']} {alumni['graduation_year']})")

c:\Users\nyong\School\Spring 1\Agentic AI\Assignments\Assignment two\alumni-rag-agent\src\tools\tavily_search.py:12: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_wrapper = TavilySearchResults(
[2026-02-17 19:34:27] INFO: Initializing Alumni RAG Agent...
[2026-02-17 19:34:32] INFO: ✓ Retrieval module initialized
[2026-02-17 19:34:32] INFO: ✓ Tools initialized: email_sender, linkedin_scraper, survey_tool, linkedin_discovery
[2026-02-17 19:34:33] INFO: ✓ Verification module initialized
[2026-02-17 19:34:33] INFO: ✓ ReAct agent initialized
[2026-02-17 19:34:33] INFO: Alumni RAG Agent ready!



Sample alumni data available: 3 profiles
  - John Doe (MSIT 2023)
  - Jane Smith (MSECE 2023)
  - Alice Wanjiku (MSIT 2022)


## 3. Ingest Data (Manual & Automated)

In [5]:
#Option A: Ingest sample alumni profiles (Manual)
# chunk_count = agent.ingest_alumni(SAMPLE_ALUMNI)
# print(f"Ingested {chunk_count} document chunks from sample data")

# Option B: Automated Discovery (Tavily Search -> Extract -> Scrape -> Ingest)
# Only runs if Tavily API keys are present
if os.environ.get("Tavily_API_KEY"):
    print("\nStarting automated discovery for MSIT 2023...")
    discovered = agent.discover_and_ingest(program="MSIT", year=2023)
    print(f"Discovered and ingested {len(discovered)} new profiles")
else:
    print("\nSkipping automated discovery (Tavily API Key not set)")

[2026-02-05 15:41:05] INFO: Starting automated discovery for MSIT 2023...



Starting automated discovery for MSIT 2023...


[2026-02-05 15:41:09] INFO: Found 5 search result snippets
[2026-02-05 15:41:17] INFO: HTTP Request: POST https://ai-gateway.andrew.cmu.edu/chat/completions "HTTP/1.1 200 OK"
[2026-02-05 15:41:17] INFO: Extracted 6 structured profiles via LLM
[2026-02-05 15:41:18] INFO: HTTP Request: POST https://ai-gateway.andrew.cmu.edu/embeddings "HTTP/1.1 200 OK"
[2026-02-05 15:41:18] INFO:   ✓ Ingested 1 chunks for Fabrice ITULAMYA MASUMBUKO
[2026-02-05 15:41:19] INFO: HTTP Request: POST https://ai-gateway.andrew.cmu.edu/embeddings "HTTP/1.1 200 OK"
[2026-02-05 15:41:19] INFO:   ✓ Ingested 1 chunks for James Niyongira
[2026-02-05 15:41:19] INFO: HTTP Request: POST https://ai-gateway.andrew.cmu.edu/embeddings "HTTP/1.1 200 OK"
[2026-02-05 15:41:19] INFO:   ✓ Ingested 1 chunks for Dona G. Junias BONOU
[2026-02-05 15:41:20] INFO: HTTP Request: POST https://ai-gateway.andrew.cmu.edu/embeddings "HTTP/1.1 200 OK"
[2026-02-05 15:41:20] INFO:   ✓ Ingested 1 chunks for Mike Mwanje
[2026-02-05 15:41:21] INF

Discovered and ingested 6 new profiles


## 4. Test Retrieval Module

In [7]:
# Test direct search
results = agent.search("Alumini that Graduation 2025?", k=5)

print("Search Results:")
for i, doc in enumerate(results, 1):
    print(f"\n{i}. Alumni: {doc.metadata.get('alumni_id', 'N/A')}")
    print(f"   Content: {doc.page_content[:150]}...")

[2026-02-17 19:39:53] INFO: HTTP Request: POST https://ai-gateway.andrew.cmu.edu/embeddings "HTTP/1.1 200 OK"


Search Results:

1. Alumni: james_niyongira
   Content: Alumni: James Niyongira
ID: james_niyongira
Email: james_niyongira@alumni.cmu.edu
Graduation: 2023 - Unknown
Current Position: Unknown at Unknown
Loca...

2. Alumni: james_niyongira
   Content: Alumni: James Niyongira
ID: james_niyongira
Email: james_niyongira@alumni.cmu.edu
Graduation: 2023 - Unknown
Current Position: Unknown at Unknown
Loca...

3. Alumni: maurine_gatimu
   Content: Alumni: Maurine Gatimu
ID: maurine_gatimu
Email: maurine_gatimu@alumni.cmu.edu
Graduation: 2025 - MSIT
Current Position: Unknown at Unknown
Location: ...

4. Alumni: maurine_gatimu
   Content: Alumni: Maurine Gatimu
ID: maurine_gatimu
Email: maurine_gatimu@alumni.cmu.edu
Graduation: 2025 - MSIT
Current Position: Unknown at Unknown
Location: ...

5. Alumni: fabrice_itulamya_masumbuko
   Content: Alumni: Fabrice ITULAMYA MASUMBUKO
ID: fabrice_itulamya_masumbuko
Email: fabrice_itulamya_masumbuko@alumni.cmu.edu
Graduation: 2023 - Unknown
Current ...


In [3]:
# Test direct search
results = agent.search("fintech experience", k=3)

print("Search Results:")
for i, doc in enumerate(results, 1):
    print(f"\n{i}. Alumni: {doc.metadata.get('alumni_id', 'N/A')}")
    print(f"   Content: {doc.page_content[:150]}...")

[2026-02-17 19:35:19] INFO: HTTP Request: POST https://ai-gateway.andrew.cmu.edu/embeddings "HTTP/1.1 200 OK"


Search Results:

1. Alumni: pax_elisee_mfura
   Content: Alumni: Pax Elisee Mfura
ID: pax_elisee_mfura
Email: pax_elisee_mfura@alumni.cmu.edu
Graduation: 2023 - MSIT
Current Position: Unknown at Unknown
Loca...

2. Alumni: pax_elisee_mfura
   Content: Alumni: Pax Elisee Mfura
ID: pax_elisee_mfura
Email: pax_elisee_mfura@alumni.cmu.edu
Graduation: 2023 - MSIT
Current Position: Unknown at Unknown
Loca...

3. Alumni: mike_mwanje
   Content: Alumni: Mike Mwanje
ID: mike_mwanje
Email: mike_mwanje@alumni.cmu.edu
Graduation: 2023 - MSIT
Current Position: Unknown at AirQo
Location: Pittsburgh,...


## 5. Run Agent Query (Multi-Step ReAct Loop)

In [ ]:
# Run a query that requires multiple steps
result = agent.run(
    "Find alumni who work in fintech and send them a check-in email"
)

print("\n" + "="*50)
print("FINAL RESPONSE:")
print("="*50)
print(result["response"])

[2026-02-05 06:17:25] INFO: Query: Find alumni who work in fintech and send them a check-in email
[2026-02-05 06:17:25] INFO: [ITERATION] Starting iteration 1...
[2026-02-05 06:17:27] INFO: HTTP Request: POST https://ai-gateway.andrew.cmu.edu/chat/completions "HTTP/1.1 200 OK"
[2026-02-05 06:17:27] INFO: [REASON] 1. What information do I need?
   - I need to identify alumni who work in the fintech industry from ...
[2026-02-05 06:17:27] INFO: DECISION POINT: Options=['RETRIEVE', 'email_sender', 'linkedin_scraper', 'survey_tool', 'FINAL_ANSWER'], Selected=RETRIEVE
[2026-02-05 06:17:27] INFO: [ACT] Executing RETRIEVAL_MODULE...
[2026-02-05 06:17:28] INFO: HTTP Request: POST https://ai-gateway.andrew.cmu.edu/embeddings "HTTP/1.1 200 OK"
[2026-02-05 06:17:28] INFO: [ITERATION] Starting iteration 2...
[2026-02-05 06:17:30] INFO: HTTP Request: POST https://ai-gateway.andrew.cmu.edu/chat/completions "HTTP/1.1 200 OK"
[2026-02-05 06:17:30] INFO: [REASON] 1. What information do I need?
   - I n


FINAL RESPONSE:
Based on the retrieved context, the alumni who currently work in fintech is Alice Wanjiku. She is a Product Manager at FinTech Kenya. An email has been successfully sent to her at nyonggodwill@gmail.com as part of the check-in process. If you need further assistance or wish to contact other alumni, please let me know!


## 6. View Verification Results

In [ ]:
verification = result["verification"]

print("Verification Report:")
print(f"  Score: {verification.score:.2f}")
print(f"  Confidence: {verification.confidence}")
print(f"  Recommendation: {verification.recommendation}")

print("\nClaims Analysis:")
for i, claim in enumerate(verification.claims, 1):
    status = "✓" if claim.verified else "✗"
    print(f"  {i}. [{status}] {claim.claim[:60]}...")

Verification Report:
  Score: 1.00
  Confidence: high
  Recommendation: proceed

Claims Analysis:
  1. [✓] Based on the retrieved context, the alumni who currently wor...


## 7. View Execution Trace

In [ ]:
print("Execution Trace:")
for step in result["trace"]:
    print(f"  [{step['phase']}] ", end="")
    if step['phase'] == 'REASON':
        print(f"{step['thought'][:80]}...")
    elif step['phase'] == 'DECIDE':
        print(f"Decision: {step['decision']}")
    elif step['phase'] == 'ACT':
        print(f"Module: {step.get('module', 'N/A')} - {step.get('result', step.get('tool', ''))}")
    elif step['phase'] == 'VERIFY':
        print(f"Score: {step['score']:.2f}, Confidence: {step['confidence']}")

Execution Trace:
  [REASON] 1. What information do I need?
   - I need to identify alumni who work in the fi...
  [DECIDE] Decision: RETRIEVE
  [ACT] Module: RETRIEVAL_MODULE - Retrieved 3 documents
  [REASON] 1. What information do I need?
   - I need to identify alumni who are currently ...
  [DECIDE] Decision: email_sender
  [ACT] Module: TOOL_MODULE - {'success': True, 'message_id': 'msg_1770265055.917241', 'recipient': 'nyonggodwill@gmail.com', 'err
  [REASON] To address the query, "Find alumni who work in fintech and send them a check-in ...
  [DECIDE] Decision: email_sender
  [ACT] Module: TOOL_MODULE - {'success': True, 'message_id': 'msg_1770265062.157903', 'recipient': 'nyonggodwill@gmail.com', 'err
  [REASON] To address the query, I need to identify alumni who work in fintech and send the...
  [DECIDE] Decision: email_sender
  [ACT] Module: TOOL_MODULE - {'success': True, 'message_id': 'msg_1770265069.434187', 'recipient': 'nyonggodwill@gmail.com', 'err
  [REASON] To address

## 8. View LangSmith Trace

After running the cells above, view your traces at:

**https://smith.langchain.com/**

Look for project: `alumni-rag-agent-hw2`

In [ ]:
print(f"LangSmith Project: {os.environ.get('LANGCHAIN_PROJECT', 'Not set')}")
print("\nView your traces at: https://smith.langchain.com/")
print("Export traces for HW2 deliverables from the LangSmith dashboard.")

LangSmith Project: alumni-rag-agent-hw2

View your traces at: https://smith.langchain.com/
Export traces for HW2 deliverables from the LangSmith dashboard.


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Your existing LLM setup
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    base_url='https://ai-gateway.andrew.cmu.edu/',
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)

# Define a simple test tool
@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"Sunny in {city}"

# Try binding the tool to the LLM
llm_with_tools = llm.bind_tools([get_weather])

# Invoke it — if function calling works, the response will have tool_calls
response = llm_with_tools.invoke("What's the weather in Pittsburgh?")

print("Tool calls:", response.tool_calls)  # Should show the tool call
print("Content:", response.content)        # May be empty (model chose tool instead)

Tool calls: [{'name': 'get_weather', 'args': {'city': 'Pittsburgh'}, 'id': 'call_CHiYvP8lIbjQOZ83jwPHklXG', 'type': 'tool_call'}]
Content: 
